In [2]:
%pip install reportlab

In [3]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PI
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak

# ---- 1. Load and profile ----
df = pd.read_csv("data.csv"); T = "Overtraining"
feats = [c for c in df.columns if c != T]
corr = df.corr(); vc = df[T].value_counts().sort_index()
print(df.shape, df.isna().sum().sum(), df.duplicated().sum())
print(df.describe().T.round(2)); print(corr[T].sort_values().round(3))
print(df.groupby(T).mean().T.round(2))

# ---- 2. Charts ----
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(["0 (No)", "1 (Yes)"], vc.values, color=["#4C78A8", "#E45756"])
for i, v in enumerate(vc.values): ax.text(i, v + 5, f"{v} ({v/len(df):.0%})", ha="center")
ax.set_title("Overtraining class balance"); fig.tight_layout(); fig.savefig("balance.png", dpi=150); plt.close(fig)

fig, axes = plt.subplots(4, 4, figsize=(9, 8))
for ax, c in zip(axes.flat, df.columns):
    ax.hist(df[c], bins=20, color="#4C78A8"); ax.set_title(c, fontsize=8); ax.tick_params(labelsize=6)
fig.tight_layout(); fig.savefig("hist.png", dpi=150); plt.close(fig)

fig, axes = plt.subplots(3, 5, figsize=(9, 6.5))
for ax, c in zip(axes.flat, feats):
    ax.boxplot([df[df[T] == 0][c], df[df[T] == 1][c]], tick_labels=["0", "1"])
    ax.set_title(c, fontsize=8); ax.tick_params(labelsize=7)
fig.suptitle("Features by Overtraining (0 = No, 1 = Yes)"); fig.tight_layout()
fig.savefig("box.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7)
fig.colorbar(im, shrink=0.8); ax.set_title("Correlation matrix")
fig.tight_layout(); fig.savefig("corr.png", dpi=150); plt.close(fig)

tc = corr[T].drop(T).sort_values()
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.barh(tc.index, tc.values, color=np.where(tc.values > 0, "#E45756", "#4C78A8"))
ax.set_title("Correlation of each feature with Overtraining")
fig.tight_layout(); fig.savefig("target_corr.png", dpi=150); plt.close(fig)

# ---- 3. PDF helpers ----
ss = getSampleStyleSheet()
def tbl(data, colw=None):
    t = Table(data, colWidths=colw, repeatRows=1)
    t.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,0), colors.HexColor("#2F4B7C")),
        ("TEXTCOLOR", (0,0), (-1,0), colors.white), ("FONTSIZE", (0,0), (-1,-1), 8),
        ("GRID", (0,0), (-1,-1), 0.25, colors.grey),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, colors.HexColor("#F2F4F8")])]))
    return t
def img(p, w):
    iw, ih = PI.open(p).size
    return Image(p, width=w, height=w * ih / iw)

# ---- 4. Report content ----
s = [Paragraph("Dataset Summary Report", ss["Title"]), Paragraph("Source file: data.csv", ss["Normal"]),
     Spacer(1, 12), Paragraph("1. Overview", ss["Heading1"])]
s.append(tbl([["Metric", "Value"], ["Rows", len(df)], ["Columns", df.shape[1]],
    ["Missing values", int(df.isna().sum().sum())], ["Duplicate rows", int(df.duplicated().sum())],
    ["Target column", T], ["Overtraining = 0", f"{vc[0]} ({vc[0]/len(df):.0%})"],
    ["Overtraining = 1", f"{vc[1]} ({vc[1]/len(df):.0%})"]], [2.5*inch, 2.5*inch]))
s += [Spacer(1, 10), Paragraph("Key findings", ss["Heading2"])]
for b in ["The data is clean: no missing values and no duplicate rows.",
          "Classes are moderately imbalanced (60% vs 40%).",
          "Strongest links to Overtraining: TrainingHoursPerWeek (+0.53), RestHoursPerWeek (-0.47), "
          "TrainingIntensity (+0.26), SleepQualityScore (-0.17).",
          "Biomarkers (cortisol, testosterone, CK, heart rate) show near-zero linear correlation."]:
    s.append(Paragraph("&bull; " + b, ss["Normal"]))
s.append(PageBreak())

s += [Paragraph("2. Column types", ss["Heading1"])]
s.append(tbl([["Column", "Type", "Unique values"]] + [[c, str(df[c].dtype), df[c].nunique()] for c in df.columns],
             [2.5*inch, 1.5*inch, 1.5*inch])); s.append(PageBreak())

s += [Paragraph("3. Descriptive statistics", ss["Heading1"])]
d = df.describe().T.round(2).reset_index()
s.append(tbl([["Column", "count", "mean", "std", "min", "25%", "50%", "75%", "max"]] + d.values.tolist(),
             [1.6*inch] + [0.65*inch]*8)); s.append(PageBreak())

s += [Paragraph("4. Feature means by Overtraining", ss["Heading1"])]
g = df.groupby(T).mean().T.round(2); g["diff"] = (g[1] - g[0]).round(2)
s.append(tbl([["Feature", "No (0)", "Yes (1)", "Difference"]] + g.reset_index().values.tolist(),
             [2.2*inch, 1.2*inch, 1.2*inch, 1.2*inch]))
s += [Spacer(1, 12), img("balance.png", 4.5*inch), PageBreak()]
s += [Paragraph("5. Distributions", ss["Heading1"]), img("hist.png", 6.8*inch), PageBreak()]
s += [Paragraph("6. Features by Overtraining", ss["Heading1"]), img("box.png", 6.8*inch), PageBreak()]
s += [Paragraph("7. Correlations", ss["Heading1"]), img("corr.png", 5.6*inch), img("target_corr.png", 4.2*inch)]

# ---- 5. Save ----
SimpleDocTemplate("summary_report.pdf", pagesize=letter, leftMargin=0.7*inch, rightMargin=0.7*inch,
                  topMargin=0.7*inch, bottomMargin=0.7*inch).build(s)
print("Saved summary_report.pdf")

(600, 16) 0 0
                       count    mean     std     min     25%     50%     75%  \
Age                    600.0   23.93    3.82   16.00   21.00   24.00   26.00   
TrainingHoursPerWeek   600.0   18.52    5.82    5.00   14.63   18.63   22.36   
TrainingIntensity      600.0    6.01    1.73    3.02    4.45    6.08    7.54   
RestHoursPerWeek       600.0   30.12    9.47   10.00   23.81   29.90   36.57   
SleepQualityScore      600.0    6.01    1.70    3.00    4.52    6.14    7.40   
HeartRateRest          600.0   64.70    9.86   45.00   57.78   64.40   71.66   
HeartRateVar           600.0   69.53   20.07   20.00   55.39   69.07   81.86   
CortisolLevel          600.0   12.77    3.89    5.00   10.15   12.92   15.35   
TestosteroneLevel      600.0  601.56  147.70  200.00  502.29  602.23  700.41   
CKLevel                600.0  392.64  186.42   80.00  266.20  381.05  520.47   
MoodScore              600.0    5.86    1.69    3.00    4.40    5.84    7.17   
ReactionTime           600

<ipython-input-3-ceec3f9659ee>:22: MatplotlibDeprecationWarning: The x parameter as float was deprecated in Matplotlib 3.10 and will be removed in 3.12. Use int(x) instead.
  ax.set_title("Overtraining class balance"); fig.tight_layout(); fig.savefig("balance.png", dpi=150); plt.close(fig)
<ipython-input-3-ceec3f9659ee>:22: MatplotlibDeprecationWarning: The y parameter as float was deprecated in Matplotlib 3.10 and will be removed in 3.12. Use int(y) instead.
  ax.set_title("Overtraining class balance"); fig.tight_layout(); fig.savefig("balance.png", dpi=150); plt.close(fig)
<ipython-input-3-ceec3f9659ee>:27: MatplotlibDeprecationWarning: The x parameter as float was deprecated in Matplotlib 3.10 and will be removed in 3.12. Use int(x) instead.
  fig.tight_layout(); fig.savefig("hist.png", dpi=150); plt.close(fig)
<ipython-input-3-ceec3f9659ee>:27: MatplotlibDeprecationWarning: The y parameter as float was deprecated in Matplotlib 3.10 and will be removed in 3.12. Use int(y) instead.
 

Saved summary_report.pdf
